In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nfl_data_py as nfl
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from scipy import stats
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/processed/merged_data_with_college.csv')

In [2]:
df_wr = df[df['pos_group'] == 'WR']
for pct in [70, 75, 80, 85]:
    val = df_wr['w_av'].quantile(pct/100)
    print(f"{pct}th percentile: {val}")

70th percentile: 19.0
75th percentile: 23.0
80th percentile: 27.800000000000068
85th percentile: 35.60000000000002


In [3]:
print(df[(df['w_av'] >= 32) & (df['pos'] == 'WR')][['player_name', 'college', 'pos', 'w_av', 'pick']])

          player_name        college pos  w_av  pick
271   Plaxico Burress   Michigan St.  WR  70.0     8
273  Laveranues Coles    Florida St.  WR  68.0    78
279   Darrell Jackson        Florida  WR  55.0    80
282  Dennis Northcutt        Arizona  WR  41.0    32
285      Jerry Porter  West Virginia  WR  33.0    47
..                ...            ...  ..   ...   ...
842      Nico Collins       Michigan  WR  33.0    89
870      Drake London            USC  WR  32.0     8
877    George Pickens        Georgia  WR  36.0    52
894       Zay Flowers    Boston Col.  WR  33.0    22
903        Puka Nacua            BYU  WR  39.0   177

[121 rows x 5 columns]


## Going with w_av >= 32 for WRs. Lots of guys that are your glue guys.

In [4]:
df_wr['is_hit'] = (df_wr['w_av'] >= 32).astype(int)
print(f"Hits: {df_wr['is_hit'].sum():.0f}, Busts: {(df_wr['is_hit'] == 0).sum()}")

Hits: 121, Busts: 583


In [5]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
df_wr_clean = df_wr.dropna(subset=features + ['is_hit'])
print(f"WRs with all features: {len(df_wr_clean)}")
X = df_wr_clean[features]
y = df_wr_clean['is_hit']

print(f"Shape: {X.shape}")
print(y.value_counts())

WRs with all features: 535
Shape: (535, 5)
is_hit
0    452
1     83
Name: count, dtype: int64


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.507
              precision    recall  f1-score   support

           0       0.81      0.54      0.65       113
           1       0.12      0.33      0.17        21

    accuracy                           0.51       134
   macro avg       0.47      0.44      0.41       134
weighted avg       0.70      0.51      0.57       134

Accuracy: 0.836
              precision    recall  f1-score   support

           0       0.84      0.99      0.91       113
           1       0.00      0.00      0.00        21

    accuracy                           0.84       134
   macro avg       0.42      0.50      0.46       134
weighted avg       0.71      0.84      0.77       134

Accuracy: 0.784
              precision    recall  f1-score   support

           0       0.84      0.92      0.88       113
           1       0.10      0.05      0.06        21

    accuracy                           0.78       134
   macro avg       0.47      0.48      0.47       134
weighted avg       0.72   

## Random forest had a really high accuracy but really didn't find any of the actual hits. Implying that it probably just predicted bust for most of the guys.

## Gradient boosting had good accuracy but low hit recall, and only got it correct about 10% of the time.

In [7]:
college_features = ['career_g', 'career_rec', 'career_rec_yds', 'career_rec_yr',
                    'career_rec_td', 'career_rec_yg',
                    'last_g', 'last_rec', 'last_rec_yds', 'last_rec_yr',
                    'last_rec_td', 'last_rec_yg']

all_features = features + college_features
df_wr_college = df_wr.dropna(subset=all_features + ['is_hit'])
print(f"WRs with all features: {len(df_wr_college)}")

WRs with all features: 426


In [8]:
X = df_wr_college[all_features]
y = df_wr_college['is_hit']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size= 0.25,
                                                    random_state=42,
                                                    stratify=y)

## standardizing data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## running LogisiticRegression
model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(classification_report(y_test,y_pred))

## running RandomForest
rf_model = RandomForestClassifier(random_state=42, class_weight='balanced')
rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_pred):.3f}")
print(classification_report(y_test,rf_pred))

## running GradientBoost
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_scaled, y_train)
gb_pred = gb_model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, gb_pred):.3f}")
print(classification_report(y_test, gb_pred))

for name, mod in [('LogReg', LogisticRegression(random_state=42, class_weight='balanced')),
                   ('RF', RandomForestClassifier(random_state=42, class_weight='balanced')),
                   ('GB', GradientBoostingClassifier(random_state=42))]:
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='accuracy')
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

Accuracy: 0.598
              precision    recall  f1-score   support

           0       0.90      0.59      0.71        90
           1       0.23      0.65      0.34        17

    accuracy                           0.60       107
   macro avg       0.56      0.62      0.52       107
weighted avg       0.79      0.60      0.65       107

Accuracy: 0.841
              precision    recall  f1-score   support

           0       0.85      0.99      0.91        90
           1       0.50      0.06      0.11        17

    accuracy                           0.84       107
   macro avg       0.67      0.52      0.51       107
weighted avg       0.79      0.84      0.78       107

Accuracy: 0.804
              precision    recall  f1-score   support

           0       0.85      0.93      0.89        90
           1       0.25      0.12      0.16        17

    accuracy                           0.80       107
   macro avg       0.55      0.53      0.52       107
weighted avg       0.75   

### really good increased across the board, especially for CV. going to see if the changing the prediction threshold can give me a higher precision and recall on GB

In [9]:
## Adjust prediction threshold for GB
probs = gb_model.predict_proba(X_test_scaled)[:, 1]

for thresh in [0.50, 0.40, 0.30, 0.20]:
    pred = (probs >= thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(classification_report(y_test, pred))


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.85      0.93      0.89        90
           1       0.25      0.12      0.16        17

    accuracy                           0.80       107
   macro avg       0.55      0.53      0.52       107
weighted avg       0.75      0.80      0.77       107


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.86      0.92      0.89        90
           1       0.30      0.18      0.22        17

    accuracy                           0.80       107
   macro avg       0.58      0.55      0.55       107
weighted avg       0.77      0.80      0.78       107


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.88      0.89      0.88        90
           1       0.38      0.35      0.36        17

    accuracy                           0.80       107
   macro avg       0.63      0.62      0.62       107
weighted avg       0.80   

### Threshold of 0.3 is good for WR, has the highest hit recall and solid hit precision. It predicts hit more often than the other 2 models and gets it right about 40% of the time. which for WR isn't bad. teams usually draft multiple WRs so having the highest chance of them being a hit matters

In [10]:
features = ['ht', 'wt', 'forty', 'vertical', 'broad_jump']
college_features = ['career_g', 'career_rec', 'career_rec_yds', 'career_rec_yr',
                    'career_rec_td', 'career_rec_yg',
                    'last_g', 'last_rec', 'last_rec_yds', 'last_rec_yr',
                    'last_rec_td', 'last_rec_yg']

all_features = features + college_features

df_wr_draft = df_wr.dropna(subset=all_features + ['pick'])
X = df_wr_draft[all_features]
y = df_wr_draft['pick']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}\n")

for name, mod in [('Ridge', Ridge(random_state=42)),
                   ('Lasso', Lasso(random_state=42)),
                   ('RF', RandomForestRegressor(random_state=42)),
                   ('GB', GradientBoostingRegressor(random_state=42))]:
    mod.fit(X_train_scaled, y_train)
    pred = mod.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name}:")
    print(f"  MAE: {mae:.1f} picks off")
    print(f"  RMSE: {rmse:.1f}")
    print(f"  R²: {r2:.3f}")
    scores = cross_val_score(mod, X_train_scaled, y_train, cv=5, scoring='neg_mean_absolute_error')
    print(f"  CV MAE: {-scores.mean():.1f} (+/- {scores.std():.1f})\n")

Train: (319, 17), Test: (107, 17)

Ridge:
  MAE: 51.5 picks off
  RMSE: 63.5
  R²: 0.228
  CV MAE: 51.7 (+/- 6.3)

Lasso:
  MAE: 52.0 picks off
  RMSE: 64.3
  R²: 0.207
  CV MAE: 50.7 (+/- 6.6)

RF:
  MAE: 53.8 picks off
  RMSE: 66.3
  R²: 0.157
  CV MAE: 54.2 (+/- 5.9)

GB:
  MAE: 54.3 picks off
  RMSE: 68.4
  R²: 0.104
  CV MAE: 53.2 (+/- 6.7)



## Ridge is best draft prediction model

In [11]:
X_all = df_wr_college[all_features]
y_hit = df_wr_college['is_hit']
y_pick = df_wr_college['pick']

scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

## Hit probability
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_all_scaled, y_hit)
hit_prob = gb_model.predict_proba(X_all_scaled)[:, 1]

## Predicted pick
ridge_model = Ridge(random_state=42)
ridge_model.fit(X_all_scaled, y_pick)
pred_pick = ridge_model.predict(X_all_scaled)

## Combine results
results = df_wr_college[['player_name', 'college', 'season', 'pick', 'w_av', 'is_hit']].copy()
results['hit_probability'] = hit_prob.round(3)
results['predicted_pick'] = pred_pick.round(1)
results['actual_pick'] = results['pick']
results['pick_difference'] = results['actual_pick'] - results['predicted_pick']

## Top prospects by hit probability
print("TOP WR prospects (highest hit probability):")
print(results[results['predicted_pick'] > 0].sort_values('hit_probability', ascending=False).head(20)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av']
].to_string(index=False))

TOP WR prospects (highest hit probability):
     player_name        college  season  actual_pick  predicted_pick  hit_probability  w_av
    Amari Cooper        Alabama    2015            4            50.7            0.964  66.0
   Jarvis Landry            LSU    2014           63           143.2            0.937  56.0
   Davante Adams     Fresno St.    2014           53            50.6            0.935  92.0
   Brandin Cooks     Oregon St.    2014           20            30.5            0.920  73.0
  George Pickens        Georgia    2022           52           105.6            0.889  36.0
   Nate Burleson         Nevada    2003           71            83.7            0.878  45.0
  Brandon LaFell            LSU    2010           78           155.2            0.864  43.0
   Tyler Lockett     Kansas St.    2015           69            57.1            0.856  68.0
    D.K. Metcalf    Mississippi    2019           64            26.6            0.853  57.0
   Sammy Watkins        Clemson    2

In [12]:
results['value_score'] = results['hit_probability'] * 100 + results['pick_difference']
print("\nMOST UNDERVALUED WRs (hit probability > 0.3 AND drafted later than predicted):")
undervalued = results[(results['hit_probability'] > 0.3) & (results['pick_difference'] > 0) & (results['predicted_pick'] > 0)]
print(undervalued.sort_values('value_score', ascending=False)[
    ['player_name', 'college', 'season', 'actual_pick', 'predicted_pick', 'hit_probability', 'w_av', 'value_score']
].head(15).to_string(index=False))


MOST UNDERVALUED WRs (hit probability > 0.3 AND drafted later than predicted):
     player_name           college  season  actual_pick  predicted_pick  hit_probability  w_av  value_score
   Darren Waller      Georgia Tech    2015          204           103.5            0.647  33.0        165.2
    D.K. Metcalf       Mississippi    2019           64            26.6            0.853  57.0        122.7
Justin McCareins Northern Illinois    2001          124            67.1            0.519  33.0        108.8
  Darnell Mooney            Tulane    2020          173           130.6            0.633  33.0        105.7
   Tyler Lockett        Kansas St.    2015           69            57.1            0.856  68.0         97.5
   Davante Adams        Fresno St.    2014           53            50.6            0.935  92.0         95.9
  Darius Slayton            Auburn    2019          171           119.1            0.421  32.0         94.0
   Greg Jennings  Western Michigan    2006           52 